# Use-case: Golf Forecasting

Generate a forecasting dataset about professional golf (tournaments, majors, rankings) using the LightningRod SDK. This example showcases dataset generation, preparation with SDK utils, and training results from our experiments.

In [1]:
%pip install lightningrod-ai python-dotenv pandas

from IPython.display import clear_output
clear_output()

from datetime import datetime

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

## Set up the client

Sign up at [dashboard.lightningrod.ai](https://dashboard.lightningrod.ai/?redirect=/api) to get your API key and **$50 of free credits**.

In [2]:
from lightningrod import LightningRod
from lightningrod.utils import config

api_key = config.get_config_value("LIGHTNINGROD_API_KEY")
lr = LightningRod(api_key=api_key)

## Build the pipeline

Configure the pipeline with domain-specific instructions and examples for golf forecasting.

In [3]:
instructions = """
Generate binary forecasting questions about professional golf across all major tours and events.

Cover what golf fans bet on: tournament outcomes, cuts, matchups, majors, team events, season races, world rankings, and player milestones.

Questions should be specific, verifiable, and span the full probability spectrum.
"""

good_examples = [
    "Will Scottie Scheffler win the 2025 Masters?",
    "Will the 2025 US Open winning score be under par?",
    "Will Tiger Woods make the cut at the 2025 Masters?",
    "Will Rory McIlroy finish top 5 at the 2025 US Open?",
    "Will any LIV player win a major championship in 2025?",
    "Will Europe win the 2025 Ryder Cup?",
    "Will any player win 4+ PGA Tour events in 2025?",
    "Will Scottie Scheffler remain world #1 through June 2025?",
    "Will a first-time major winner emerge at the 2025 PGA Championship?",
    "Will Nelly Korda win the 2025 US Women's Open?",
]

bad_examples = [
    "Will someone win the tournament? (obvious)",
    "Will golf be exciting? (subjective)",
    "Will there be birdies? (trivial)",
]

search_queries = [
    "PGA Tour",
    "LIV Golf",
    "LPGA",
    "golf major championship",
    "Ryder Cup Presidents Cup",
    "golf world rankings",
    "professional golf",
    "women's golf",
    "European Tour golf",
]

In [4]:
from lightningrod import (
    BinaryAnswerType,
    NewsSeedGenerator,
    ForwardLookingQuestionGenerator,
    NewsContextGenerator,
    WebSearchLabeler,
    QuestionPipeline,
)

answer_type = BinaryAnswerType()

pipeline = QuestionPipeline(
    seed_generator=NewsSeedGenerator(
        start_date=datetime(2024, 6, 1),
        end_date=datetime(2026, 1, 1),
        interval_duration_days=14,
        search_query=search_queries,
        articles_per_search=10,
    ),
    question_generator=ForwardLookingQuestionGenerator(
        instructions=instructions,
        examples=good_examples,
        bad_examples=bad_examples,
        answer_type=answer_type,
        questions_per_seed=5,
    ),
    context_generators=[
        NewsContextGenerator(
            articles_per_query=3,
            num_search_queries=3,
            num_articles=5,
        )
    ],
    labeler=WebSearchLabeler(answer_type=answer_type),
)

## Run the pipeline

This will collect news articles, generate questions, and find answers. Use `max_questions` to limit the run for testing.

In [5]:
dataset = lr.transforms.run(pipeline, max_questions=100, name="Golf forecasting")
samples = dataset.download()

pct = (sum(1 for s in samples if s.is_valid is True) / len(samples) * 100) if samples else 0
print(f"{len(samples)} samples ({pct:.1f}% valid)")

Output()

100 samples (87.0% valid)


## Prepare the dataset

Use SDK utils to filter valid samples, deduplicate, and split into train/test sets. We filter by `date_close <= today` to only include questions that have already resolved.

In [ ]:
from lightningrod.training import prepare_for_training

train, test = prepare_for_training(
    samples,
    answer_type,
    test_size=0.2,
    split_strategy="temporal",
    include_assistant=True,
    days_to_resolution_range=(1, None), # at least 1 day to resolution
)

for name, data in [("Train", train), ("Test", test)]:
    yes_count = sum(s["label"] for s in data)
    print(f"{name}: {len(data)} rows, {yes_count/len(data)*100:.1f}% yes")

Train: 37 rows, 32.4% yes
Test: 18 rows, 44.4% yes


In [7]:
def _display_head(data, name, n=5):
    if not data:
        print(f"{name}: no rows")
        return
    df = pd.DataFrame(data[:n])
    cols = [
        "question_text", "prediction_date", "date_close", "resolution_date",
        "answer_type", "answer_parser_type", "reward_function_type",
        "label","label_confidence", "prompt"
    ]
    display_cols = [c for c in cols if c in df.columns]
    print(f"{name} (head):")
    display(df[display_cols])

_display_head(train, "Train")
_display_head(test, "Test")

Train (head):


,question_text,prediction_date,date_close,resolution_date,answer_type,label,label_confidence,prompt
0,Will a player from the UCLA lineup (Fiorellini...,2025-02-22T00:00:00,2025-02-26T00:00:00,2025-02-25T00:00:00,binary,1.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will a..."
1,"Will any player from Alabama State, North Caro...",2025-02-22T00:00:00,2025-02-25T00:00:00,2025-02-24T00:00:00,binary,1.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will a..."
2,Will the Howard University women's golf team w...,2025-02-22T00:00:00,2025-02-25T00:00:00,2025-02-24T00:00:00,binary,1.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will t..."
3,Will UCLA's Francesca Fiorellini make the 36-h...,2025-02-22T00:00:00,2025-04-04T00:00:00,2025-04-03T00:00:00,binary,0.0,0.9,"[{'role': 'user', 'content': 'QUESTION: Will U..."
4,Will the UCLA women's golf team win the team t...,2025-02-22T00:00:00,2025-02-26T00:00:00,2025-02-25T00:00:00,binary,1.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will t..."


Test (head):


,question_text,prediction_date,date_close,resolution_date,answer_type,label,label_confidence,prompt
0,Will any player win the 2025 LIV Golf Indianap...,2025-04-30T00:00:00,2025-08-18T00:00:00,2025-08-17T00:00:00,binary,1.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will a..."
1,Will the 'Crushers GC' team win the team compe...,2025-04-30T00:00:00,2025-08-18T00:00:00,2025-08-17T00:00:00,binary,0.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will t..."
2,Will Jon Rahm win the 2025 LIV Golf Indianapol...,2025-04-30T00:00:00,2025-08-18T00:00:00,2025-08-17T00:00:00,binary,0.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will J..."
3,Will Tiger Woods finish in the top 20 of any p...,2025-05-01T00:00:00,2025-09-01T00:00:00,2025-08-31T00:00:00,binary,0.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will T..."
4,Will a player from the Crushers GC team win th...,2025-05-01T00:00:00,2025-05-04T00:00:00,2025-05-04T00:00:00,binary,1.0,1.0,"[{'role': 'user', 'content': 'QUESTION: Will a..."


## Uploading the dataset to HuggingFace

Once we have a training-ready dataset, we can push it to Hugging Face for sharing or downstream use.

In [8]:
%pip install datasets -q

from datasets import Dataset, DatasetDict
from lightningrod.utils import config

dataset = DatasetDict({
    "train": Dataset.from_list(train),
    "test": Dataset.from_list(test),
})
print(f"Train: {len(dataset['train'])} rows, Test: {len(dataset['test'])} rows")
print("Columns:", dataset["train"].column_names[:8], "...")

DATASET_PATH = f"{config.get_config_value('HF_USERNAME')}/golf-forecasting-demo"
dataset.push_to_hub(DATASET_PATH, token=config.get_config_value("HF_ACCESS_TOKEN"))


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Train: 37 rows, Test: 18 rows
Columns: ['question_text', 'date_close', 'event_date', 'resolution_criteria', 'prediction_date', 'label', 'answer_type', 'label_confidence'] ...


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/bart/golf-forecasting-demo/commit/75aa82cb9a4bc3ff669cf29a4d2e0f260f6aac58', commit_message='Upload dataset', commit_description='', oid='75aa82cb9a4bc3ff669cf29a4d2e0f260f6aac58', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/bart/golf-forecasting-demo', endpoint='https://huggingface.co', repo_type='dataset', repo_id='bart/golf-forecasting-demo'), pr_revision=None, pr_num=None)

## Model Training

We used the generated dataset above to fine-tune a forecasting model via RL on 3,178 forecasting questions, surpassing GPT-5 performance.

**For more details on methods, results, and data:**
- **[Golf-Forecaster Model](https://huggingface.co/LightningRodLabs/Golf-Forecaster)**
- **[Golf-Forecaster Dataset](https://huggingface.co/datasets/LightningRodLabs/GolfForecasting)**

![Brier Skill Score](https://huggingface.co/datasets/LightningRodLabs/GolfForecasting/resolve/main/brier_skill_score.png)

**Coming Soon:** Seamlessly generate datasets, fine-tune, and evaluate your own forecasting models end-to-end on the Lightningrod platform.
 
👉 [Sign up to get early access and updates.](https://lightningrod.ai/)